# SpaceX Falcon 9 — Data Wrangling

Starting from the collected launch table (`dataset_part_1.csv`), this notebook
inspects missing values, summarizes launch sites / orbits / outcomes, and derives
the binary **landing-success label** (`Class`) used by every later notebook.

In [1]:
import pandas as pd
import numpy as np

BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"
df = pd.read_csv(f"{BASE}/datasets/dataset_part_1.csv")
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [2]:
(df.isnull().sum() / len(df) * 100).round(2)

FlightNumber       0.00
Date               0.00
BoosterVersion     0.00
PayloadMass        0.00
Orbit              0.00
LaunchSite         0.00
Outcome            0.00
Flights            0.00
GridFins           0.00
Reused             0.00
Legs               0.00
LandingPad        28.89
Block              0.00
ReusedCount        0.00
Serial             0.00
Longitude          0.00
Latitude           0.00
dtype: float64

In [3]:
df.dtypes

FlightNumber        int64
Date                  str
BoosterVersion        str
PayloadMass       float64
Orbit                 str
LaunchSite            str
Outcome               str
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad            str
Block             float64
ReusedCount         int64
Serial                str
Longitude         float64
Latitude          float64
dtype: object

## Launch site and orbit counts

In [4]:
df['LaunchSite'].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

In [5]:
df['Orbit'].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

## Landing outcome label

`Outcome` mixes the landing success flag and the landing type (e.g. `True ASDS`, `False Ocean`, `None None`). We map the *unsuccessful* outcomes to `0` and everything else to `1`.

In [6]:
df['Outcome'].value_counts()

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

In [7]:
bad_outcomes = set(df['Outcome'][[1, 3, 5, 6, 7]])
print("Outcomes treated as landing failures:", bad_outcomes)

landing_class = [0 if outcome in bad_outcomes else 1 for outcome in df['Outcome']]
df['Class'] = landing_class
df[['Class']].head(8)

Outcomes treated as landing failures: {'None None', 'True Ocean', 'False Ocean'}


,Class
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0


In [8]:
success_rate = df['Class'].mean()
print(f"Overall first-stage landing success rate: {success_rate:.1%}")
df.to_csv("dataset_part_2.csv", index=False)
print("Saved dataset_part_2.csv —", df.shape)

Overall first-stage landing success rate: 71.1%
Saved dataset_part_2.csv — (90, 18)
